# Diarization (NeMo)
nvidia/Nemotron-3-Diarization, up to 8 speakers. Runs on **every** wav in `data/raw_audios/`.
Per audio, results go to `outputs/diarization/<name>/nemotron/` (16 kHz wav, RTTM, timeline png, stats.json)
and chunks to `outputs/chunks/<name>/{diarization_turns,merged_speakers}/`.
Audios that already have a `stats.json` are skipped unless `FORCE = True`.
Restart the kernel after upgrading NeMo.

In [1]:
import json
import subprocess
import traceback
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf
import torch
from nemo.collections.asr.models import SortformerEncLabelModel
from silero_vad import get_speech_timestamps, load_silero_vad

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
wavs = sorted((root / "data" / "raw_audios").glob("*.wav"))
for i, p in enumerate(wavs):
    print(i, p.name)

/Users/macbook/Documents/IT/projects/tts/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


0 18-35 yosh bo'lsangiz, shu 8ta darsni eshitishingiz kerak.wav
1 2026-yilda Google'ga ishga kirish XARITASI.wav
2 28-son - GTA VI, Fransiyadagi noroziliklar, Tesla uchar avtomobili va boshqa yangiliklar tahlili.wav
3 29-son - Krinjdan tortib siyosatgacha hafta davomida sodir bo‘lgan voqealar muhokamasi.wav
4 31 - son - Birinçi besoqol, musulmon, tub amerikalik senatorlar. Amerika siyosati kök rangda.wav
5 40 daqiqada yillar davomida yo'qotganimizni qaytaramiz - Maqolim 1-son.wav
6 50 yoshida ham studentlikni tanlaydigan odam bilan suhbat.wav
7 99% odamlar buni eplay olmaydi - Iqtibos podcast #27.wav
8 AQLI PASTlar avlodi (bo'lib qolyapmiz).wav
9 Afsus qilmaslik uchun bugun nima qilish kerak - Hasanxon Yahyo Abdulmajid.wav
10 Chegaralar torayayotganini his qildim - Jamshidxon Ziyoxonov.wav
11 Dasturchilik faoliyatimni yakunlayman! - Azim Pulat Gashtak podkastida.wav
12 Ertaga bankrot bo‘lmaslik uchun hozir nima qilish kerak - Alisher Isayev.wav
13 HOZIRGI YOSHLAR NIMADAN BOSHLASHI KERA

In [2]:
ONLY = None          # None = every audio; or a list of indices from the list above, e.g. [3, 5]
FORCE = False        # True = redo audios that already have results
TEST_MINUTES = None  # e.g. 5 to only use the start of each audio
MAX_GAP_SEC = 1.0    # merge same-speaker turns when the gap is <= this
SILENCE_DBFS = -40   # energy threshold for the near-silence stat (not a universal noise detector)

In [3]:
diarizer = SortformerEncLabelModel.from_pretrained("nvidia/Nemotron-3-Diarization")
diarizer.eval()

# streaming settings, in 80 ms frames (340 = ~27 s chunks)
m = diarizer.sortformer_modules
m.chunk_len = 340
m.chunk_right_context = 40
m.fifo_len = 40
m.spkcache_update_period = 300
m.spkcache_len = 264
diarizer._check_streaming_parameters()

vad_model = load_silero_vad()

[NeMo W 2026-11-23 11:44:00 modelPT:192] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: null
    validate_manifest_paths: false
    sample_rate: 16000
    num_spks: 8
    session_len_sec: 105
    soft_targets: false
    batch_size: 42
    shuffle: true
    num_workers: 18
    pin_memory: true
    use_lhotse: false
    subsegment_mode: true
    subsegment_single_chunk_min_len_sec: 45
    subsegment_two_chunks_rate: 0.5
    subsegment_two_chunk_min_len_sec: 22
    subsegment_nspk_bias: 1.5
    subsegment_start_guard_sec: 0.25
    subsegment_min_first_spk_sec: 0.5
    subsegment_splice_silence_sec: 0.1
    
[NeMo W 2026-11-23 11:44:00 modelPT:199] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the

[NeMo I 2026-11-23 11:44:00 save_restore_connector:309] Model SortformerEncLabelModel was successfully restored from /Users/macbook/.cache/huggingface/hub/models--nvidia--Nemotron-3-Diarization/snapshots/52a432672732425e4c19765b9fec9b5810e6f4ba/Nemotron-3-Diarization.nemo.


[NeMo W 2026-11-23 11:44:00 sortformer_modules:241] spkcache_update_period (300) is less than chunk_len (340). The effective update period will be 340.


In [4]:
def to_16k_mono(src, dst):
    cmd = ["ffmpeg", "-y", "-loglevel", "error", "-i", str(src)]
    if TEST_MINUTES:
        cmd += ["-t", str(TEST_MINUTES * 60)]
    cmd += ["-ar", "16000", "-ac", "1", "-c:a", "pcm_s16le", str(dst)]
    subprocess.run(cmd, check=True)


def diarize(wav):
    segments = diarizer.diarize(audio=[str(wav)], batch_size=1)
    return sorted((float(s), float(e), spk) for s, e, spk in (line.split() for line in segments[0]))


def merge_turns(turns, max_gap):
    merged = []
    for s, e, spk in sorted(turns):
        if merged and spk == merged[-1][2] and s - merged[-1][1] <= max_gap:
            merged[-1][1] = max(merged[-1][1], e)
        else:
            merged.append([s, e, spk])
    return [tuple(t) for t in merged]


def save_rttm(turns, name, path):
    with open(path, "w") as f:
        for s, e, spk in turns:
            f.write(f"SPEAKER {name} 1 {s:.3f} {e - s:.3f} <NA> <NA> {spk} <NA> <NA>\n")


def plot_timeline(turns, title, path):
    speakers = sorted({spk for *_, spk in turns})
    fig, ax = plt.subplots(figsize=(12, 0.6 * len(speakers) + 1))
    for row, spk in enumerate(speakers):
        ax.broken_barh([(s, e - s) for s, e, x in turns if x == spk], (row - 0.4, 0.8))
    ax.set_yticks(range(len(speakers)), speakers)
    ax.set_xlabel("s")
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(path, dpi=100)
    plt.close(fig)


def export_chunks(audio, sr, turns, chunk_dir):
    chunk_dir.mkdir(parents=True, exist_ok=True)
    for old in chunk_dir.glob("*.wav"):  # drop leftovers from an earlier run
        old.unlink()
    manifest = []
    for i, (start, end, speaker) in enumerate(turns):
        a = max(0, round(start * sr))
        b = min(len(audio), round(end * sr))
        if b <= a:
            continue
        filepath = chunk_dir / f"{i:04d}_{speaker}.wav"
        sf.write(filepath, audio[a:b], sr)
        manifest.append({
            "audio_filepath": str(filepath),
            "start": start,
            "end": end,
            "duration": (b - a) / sr,
            "speaker": speaker,
        })
    with open(chunk_dir / "manifest.jsonl", "w", encoding="utf-8") as f:
        for item in manifest:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
    return len(manifest)


def diarization_stats(turns, total_seconds, res=100):
    # activity per speaker on a 1/res second grid
    n = int(np.ceil(total_seconds * res))
    activity = np.zeros(n, dtype=np.int16)
    for spk in {spk for *_, spk in turns}:
        mask = np.zeros(n, dtype=bool)
        for s, e, x in turns:
            if x == spk:
                mask[max(0, round(s * res)):min(n, round(e * res))] = True
        activity += mask
    return {
        "speech_pct": float(np.count_nonzero(activity >= 1) / n * 100),
        "overlap_pct": float(np.count_nonzero(activity >= 2) / n * 100),
    }


def vad_stats(audio, sr):
    stamps = get_speech_timestamps(
        torch.from_numpy(audio.astype(np.float32)), vad_model,
        sampling_rate=sr, threshold=0.5, speech_pad_ms=0,
    )
    speech = np.zeros(len(audio), dtype=bool)
    for t in stamps:
        speech[t["start"]:t["end"]] = True

    frame = int(sr * 0.1)
    counts = {"speech": 0, "other_sound": 0, "near_silence": 0}
    for a in range(0, len(audio), frame):
        chunk = audio[a:a + frame]
        dbfs = 20 * np.log10(max(np.sqrt(np.mean(chunk.astype(np.float64) ** 2)), 1e-10))
        if speech[a:a + frame].mean() >= 0.5:
            counts["speech"] += 1
        elif dbfs > SILENCE_DBFS:
            counts["other_sound"] += 1
        else:
            counts["near_silence"] += 1
    total = sum(counts.values())
    return {f"vad_{k}_pct": v / total * 100 for k, v in counts.items()}


def process(audio_path):
    name = audio_path.stem
    out_dir = root / "outputs" / "diarization" / name / "nemotron"
    chunks_root = root / "outputs" / "chunks" / name
    stats_path = out_dir / "stats.json"

    if stats_path.exists() and not FORCE:
        print("  already done, skipping")
        return json.loads(stats_path.read_text())

    out_dir.mkdir(parents=True, exist_ok=True)
    model_audio = out_dir / "audio_16k_mono.wav"
    to_16k_mono(audio_path, model_audio)

    turns = diarize(model_audio)
    save_rttm(turns, name, out_dir / f"{name}.rttm")
    plot_timeline(turns, audio_path.name, out_dir / "timeline.png")

    audio, sr = sf.read(model_audio, dtype="float32")
    merged = merge_turns(turns, MAX_GAP_SEC)
    n_turn_chunks = export_chunks(audio, sr, turns, chunks_root / "diarization_turns")
    n_merged_chunks = export_chunks(audio, sr, merged, chunks_root / "merged_speakers")

    total = len(audio) / sr
    per_speaker = {spk: round(sum(e - s for s, e, x in turns if x == spk), 1)
                   for spk in sorted({spk for *_, spk in turns})}
    stats = {
        "audio": audio_path.name,
        "total_seconds": round(total, 2),
        "turns": len(turns),
        "turn_chunks": n_turn_chunks,
        "merged_chunks": n_merged_chunks,
        "speaker_seconds": per_speaker,
        **diarization_stats(turns, total),
        **vad_stats(audio, sr),
    }
    stats_path.write_text(json.dumps(stats, ensure_ascii=False, indent=2))  # written last = done marker
    return stats

In [ ]:
selected = wavs if ONLY is None else [wavs[i] for i in ONLY]
results, failed = {}, {}

for n, audio_path in enumerate(selected, 1):
    print(f"[{n}/{len(selected)}] {audio_path.name}")
    try:
        results[audio_path.name] = process(audio_path)
    except Exception:
        failed[audio_path.name] = traceback.format_exc()
        print("  FAILED (continuing)")

print(f"\ndone: {len(results)}, failed: {len(failed)}")
for name, tb in failed.items():
    print("\n" + name + "\n" + tb)

[1/31] 18-35 yosh bo'lsangiz, shu 8ta darsni eshitishingiz kerak.wav
  already done, skipping
[2/31] 2026-yilda Google'ga ishga kirish XARITASI.wav
  already done, skipping
[3/31] 28-son - GTA VI, Fransiyadagi noroziliklar, Tesla uchar avtomobili va boshqa yangiliklar tahlili.wav
  already done, skipping
[4/31] 29-son - Krinjdan tortib siyosatgacha hafta davomida sodir bo‘lgan voqealar muhokamasi.wav
  already done, skipping
[5/31] 31 - son - Birinçi besoqol, musulmon, tub amerikalik senatorlar. Amerika siyosati kök rangda.wav
  already done, skipping
[6/31] 40 daqiqada yillar davomida yo'qotganimizni qaytaramiz - Maqolim 1-son.wav
  already done, skipping
[7/31] 50 yoshida ham studentlikni tanlaydigan odam bilan suhbat.wav
  already done, skipping
[8/31] 99% odamlar buni eplay olmaydi - Iqtibos podcast #27.wav
  already done, skipping
[9/31] AQLI PASTlar avlodi (bo'lib qolyapmiz).wav
  already done, skipping
[10/31] Afsus qilmaslik uchun bugun nima qilish kerak - Hasanxon Yahyo Abdulm

[NeMo W 2026-11-23 11:44:06 dataloader:1308] The following configuration keys are ignored by Lhotse dataloader: soft_label_thres,subsegment_two_chunks_rate,subsegment_min_first_spk_sec,subsegment_two_chunk_min_len_sec,subsegment_single_chunk_min_len_sec,validate_manifest_paths,subsegment_nspk_bias,subsampling_factor,subsegment_start_guard_sec,session_len_sec,subsegment_mode,subsegment_splice_silence_sec,num_spks
Diarizing: 0it [00:00, ?it/s][NeMo W 2026-11-23 11:44:06 nemo_logging:365] /Users/macbook/Documents/IT/projects/tts/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
      super().__init__(loader)
    
[NeMo W 2026-11-23 11:44:08 nemo_logging:365] /Users/macbook/Documents/IT/projects/tts/.venv/lib/python3.12/site-packages/torch/nn/attention/flex_attention.py:2621: UserWarning: flex_attention called without torch.compile() - this will use an un

In [ ]:
print(f"{'audio':60} {'min':>6} {'spk':>4} {'turns':>6} {'merged':>7} {'speech%':>8} {'overlap%':>9}")
for name, s in results.items():
    print(f"{name[:60]:60} {s['total_seconds'] / 60:6.1f} {len(s['speaker_seconds']):4d} "
          f"{s['turns']:6d} {s['merged_chunks']:7d} {s['speech_pct']:8.1f} {s['overlap_pct']:9.1f}")

audio                                                           min  spk  turns  merged  speech%  overlap%
18-35 yosh bo'lsangiz, shu 8ta darsni eshitishingiz kerak.wa  146.8    3   4006    2245     77.6       3.3
2026-yilda Google'ga ishga kirish XARITASI.wav                 30.7    7    402      32     91.9       0.0
28-son - GTA VI, Fransiyadagi noroziliklar, Tesla uchar avto   39.5    8    164      64     93.9       0.0
29-son - Krinjdan tortib siyosatgacha hafta davomida sodir b   40.9    8    115      50     93.4       0.1
31 - son - Birinçi besoqol, musulmon, tub amerikalik senator   47.8    8    119      61     93.3       0.2
40 daqiqada yillar davomida yo'qotganimizni qaytaramiz - Maq   44.0    1    930      91     81.2       0.0
50 yoshida ham studentlikni tanlaydigan odam bilan suhbat.wa  120.0    4   2904    1180     81.4       2.7
99% odamlar buni eplay olmaydi - Iqtibos podcast #27.wav       45.3    1    939     144     78.2       0.0
AQLI PASTlar avlodi (bo'lib qolyapmiz